In [ ]:
pip install -qU langchain-huggingface sentence-transformers

In [ ]:
pip install langchain   tiktoken pypdf  langchain-community

In [ ]:
pip install -qU faiss-cpu langchain-community langchain-core

In [ ]:
from langchain_core.documents import Document

# 5 create our list of 5 play documents

# Create our list of 5 player documents with rich metadata
player_docs = [
    Document(
        page_content="Lionel Messi is an Argentine professional footballer considered one of the greatest of all time, famous for his dribbling and playmaking.",
        metadata={"name": "Lionel Messi", "sport": "Football/Soccer", "active": True}
    ),
    Document(
        page_content="LeBron James is an American professional basketball player who has won multiple NBA championships and MVP awards.",
        metadata={"name": "LeBron James", "sport": "Basketball", "active": True}
    ),
    Document(
        page_content="Serena Williams is an American former professional tennis player who won 23 Grand Slam women's singles titles.",
        metadata={"name": "Serena Williams", "sport": "Tennis", "active": False}
    ),
    Document(
        page_content="Babar Azam is a Pakistani international cricketer and a top-order batter, known for his elegant cover drives.",
        metadata={"name": "Babar Azam", "sport": "Cricket", "active": True}
    ),
    Document(
        page_content="Muhammad Ali was an American professional boxer and activist, nicknamed 'The Greatest'.",
        metadata={"name": "Muhammad Ali", "sport": "Boxing", "active": False}
    )
]

print(f"Successfully created {len(player_docs)} documents")

In [ ]:
# generate embaddings
from langchain_huggingface import HuggingFaceEmbeddings


# 1. Initialize our embedding model (same as before)
embeddings_of_doc = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cpu'}, # Change to 'cuda' if using Colab GPU
    encode_kwargs={'normalize_embeddings': True}
)







In [ ]:
# 2 . Create Faiss vector database
from langchain_community.vectorstores import FAISS
vectore_store = FAISS.from_documents(player_docs,embeddings_of_doc)
vectore_store.save_local("Faiss_db")
print("Database is successfully build")

In [ ]:
# add document

vectore_store.add_documents(player_docs)

In [ ]:
from langchain_community.vectorstores import FAISS

# 1. Load the saved database from your folder
# You must pass the same embedding model you used to create it
# allow_dangerous_deserialization=True is required by LangChain because FAISS uses .pkl files.
# It is completely safe as long as you created the database yourself!
loaded_vector_store = FAISS.load_local(
    folder_path="Faiss_db",
    embeddings=embeddings_of_doc,
    allow_dangerous_deserialization=True
)

# 2. Access the documents stored inside the database's internal dictionary
stored_docs = loaded_vector_store.docstore._dict.values()

print(f"Successfully loaded {len(stored_docs)} documents from the database!\n")

# 3. Loop through and print each document's content and metadata
for i, doc in enumerate(stored_docs, 1):
    print(f"--- Document {i} ---")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}\n")

In [ ]:
# 1. Access the underlying raw FAISS index object
faiss_index = loaded_vector_store.index

# 2. Check how many embeddings (vectors) are stored
total_vectors = faiss_index.ntotal
print(f"Total embeddings stored in the index: {total_vectors}\n")

# 3. Loop through and view the actual numerical arrays
for i in range(total_vectors):
    # reconstruct(i) pulls the raw vector array out of the FAISS memory
    vector = faiss_index.reconstruct(i)

    # Let's also grab the associated document so we know WHO this math belongs to
    doc_id = loaded_vector_store.index_to_docstore_id[i]
    document = loaded_vector_store.docstore.search(doc_id)

    print(f"--- Embedding {i+1} ---")
    print(f"Belongs to Document: {document.metadata.get('name', 'Unknown')}")
    print(f"Total Dimensions: {len(vector)}")

    # We only print the first 5 numbers. Printing all 384+ numbers per document
    # would flood your Colab output screen!
    print(f"Vector Data (first 5 numbers): {vector[:5]}\n")

In [ ]:
# search docments
vectore_store.similarity_search(
    query = " who is linol Messi?",
    k=1
)

In [ ]:
vectore_store.similarity_search_with_score(
    query = " who is linol Messi?",
    k=1
)

In [ ]:
vectore_store.similarity_search_with_score(
    query = "",
    filer={'sport': 'Football/Soccer'}
)